In [1]:
!pip install torch torchvision numpy matplotlib opencv-python tqdm pandas Pillow imgaug

In [3]:
!pip install ultralytics


In [5]:
!pip install cvzone


In [1]:
from ultralytics import YOLO
import cv2

model=YOLO("../Yolo_weights/yolov8l.pt")
results=model("cars.jpg",show=True)
cv2.waitKey(0) 


image 1/1 C:\Users\e430375\Python\Computer Vision\cars.jpg: 640x576 22 cars, 4 trucks, 1119.5ms
Speed: 15.6ms preprocess, 1119.5ms inference, 12.7ms postprocess per image at shape (1, 3, 640, 576)


-1

In [7]:
pip install numpy matplotlib scikit-image filterpy


Note: you may need to restart the kernel to use updated packages.


In [9]:
# Fix matplotlib backend in sort.py
import os

# Replace 'TkAgg' with 'Agg' in sort.py
file_path = "sort.py"  # Replace with the actual path if not in the same directory
if os.path.exists(file_path):
    with open(file_path, "r") as file:
        content = file.read()

    content = content.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")

    with open(file_path, "w") as file:
        file.write(content)
    print("Backend in sort.py has been updated successfully.")
else:
    print(f"File {file_path} not found.")

# Install required dependencies
!pip install numpy matplotlib scikit-image filterpy

# Import sort.py
try:
    import sort
    print("sort.py imported successfully")
except Exception as e:
    print(f"Error importing sort.py: {e}")


Backend in sort.py has been updated successfully.
sort.py imported successfully


***
Car counter 



In [160]:
import cv2
import cvzone
import math
from ultralytics import YOLO
from sort import *

# Load YOLO model
model = YOLO("../Yolo_weights/yolov8n.pt")

# Define class names (for COCO dataset)
classNames = [
    'person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 
    'potted plant', 'bed', 'dining table', 'toilet', 'TV monitor', 'laptop', 
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 
    'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 
    'hair drier', 'toothbrush'
]

# Load video
cap = cv2.VideoCapture(r"C:\Users\e430375\Downloads\cars1.mp4")

# adding mask to the video
mask=cv2.imread('mask1.jpg')

tracker= Sort(max_age=20, min_hits=3, iou_threshold=0.3)

limits=[300,520,970,520]
totalcount=[]
# Process video frames
while True:
    success, img = cap.read()
    imgRegion=cv2.bitwise_and(img,mask)

    imgGraphics=cv2.imread('logo.png',cv2.IMREAD_UNCHANGED)

    # Check if the image was loaded and has an alpha channel
    if imgGraphics is None:
        print("Error: car_logo.jpg not found or failed to load.")
        exit()
    elif imgGraphics.shape[2] == 3:  # If the image doesn't have an alpha channel
        
    # Add a transparent alpha channel
        b, g, r = cv2.split(imgGraphics)
        alpha = np.ones(b.shape, dtype=b.dtype) * 255  # Fully opaque
        imgGraphics = cv2.merge((b, g, r, alpha))

    # Overlay the image
    img = cvzone.overlayPNG(img, imgGraphics, (0, 0))
   
    if not success:
        print("End of video or unable to read the video file.")
        break

    # Get results from the YOLO model
    results = model(imgRegion, stream=True)

    detections=np.empty((0, 5))
    for r in results:
        boxes = r.boxes
        for box in boxes:
            # Get bounding box coordinates
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            # Calculate width and height of the box
            w, h = x2 - x1, y2 - y1

            # Get confidence and class ID
            conf = math.ceil((box.conf[0] * 100)) / 100
            cls = int(box.cls[0])

            # Check if class ID is valid
            if cls < len(classNames):
                label = f'{classNames[cls]} {conf}'
              #  cvzone.putTextRect(img, label, (max(0, x1), max(35, y1)), scale=0.6, thickness=1,offset=3)
            else:
                print(f"Unknown class index: {cls}")

            curr=classNames[cls]

            if curr=="car" and conf>0.3:
                currentarray=np.array([x1,y1,x2,y2,conf])
                detections=np.vstack((detections,currentarray))
                

            # Draw the rectangle with cornerRect
           # cvzone.cornerRect(img, (x1, y1, w, h),l=9,rt=5)

    resultsTracker=tracker.update(detections)
    cv2.line(img,(limits[0],limits[1]),(limits[2],limits[3]),(0,0,255),5)
    
    for result in resultsTracker:
        x1, y1, x2, y2, id = map(int, result)
        print(result)
        w, h = x2 - x1, y2 - y1
        cvzone.cornerRect(img, (x1, y1, w, h),l=9,rt=2,colorR=(255,0,255))
        cvzone.putTextRect(img, f'{id}', (max(0, x1), max(20, y1)), scale=2, thickness=3,offset=10)

        cx,cy=x1+w//2,y1+h//2
        cv2.circle(img,(cx,cy),3,(0,0,255),cv2.FILLED)

        if limits[0]<cx<limits[1] and limits[1]-5<cy<limits[1]+5:
            if totalcount.count(id)==0:
                totalcount.append(id)
                cv2.line(img,(limits[0],limits[1]),(limits[2],limits[3]),(0,255,0),5)

     #   cvzone.putTextRect(img, f'count:{len(totalcount)}',(50,50))
    # Display the video with detections
    cv2.putText(img, str(len(totalcount)),(120,80),cv2.FONT_HERSHEY_PLAIN,5,(50,50,255),4)
    cv2.imshow("Video", img)
    cv2.imshow("Video with masked", imgRegion)

    # Break loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()



0: 384x640 7 cars, 1 bus, 85.7ms
Speed: 16.8ms preprocess, 85.7ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
[        433         353         581         475        2371]
[        763         567        1095         717        2370]
[        233         655         391         717        2369]
[        396         435         547         600        2368]
[        733         364         929         578        2367]
[        295         459         475         659        2366]

0: 384x640 7 cars, 73.4ms
Speed: 0.0ms preprocess, 73.4ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
[        433         353         581         475        2371]
[        763         567        1095         717        2370]
[        233         655         391         717        2369]
[        396         435         547         600        2368]
[        733         364         929         578        2367]
[        295         459         475         659        2366]


In [3]:
import subprocess
import os

# Define the Streamlit code
streamlit_code = """
import streamlit as st
import cv2
import cvzone
import numpy as np
import math
from ultralytics import YOLO
from sort import *

# Load YOLO model
model = YOLO("../Yolo_weights/yolov8n.pt")

# Define class names (for COCO dataset)
classNames = [
    'person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 
    'potted plant', 'bed', 'dining table', 'toilet', 'TV monitor', 'laptop', 
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 
    'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 
    'hair drier', 'toothbrush'
]

# Define Streamlit interface
st.title("Vehicle Counting Application")
st.write("Upload a video to count vehicles crossing a line.")

uploaded_video = st.file_uploader("Upload Video", type=["mp4", "avi", "mov"])

if uploaded_video is not None:
    # Save uploaded video to a temporary file
    with open("temp_video.mp4", "wb") as f:
        f.write(uploaded_video.read())

    # Initialize video capture
    cap = cv2.VideoCapture("temp_video.mp4")

    # Load mask (optional)
    mask = cv2.imread('mask1.jpg')

    # Initialize tracker and limits
    tracker = Sort(max_age=20, min_hits=3, iou_threshold=0.3)
    limits = [300, 520, 970, 520]
    totalcount = []

    # Display progress bar
    progress = st.empty()
    count_placeholder = st.empty()

    # Process the video
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for frame_id in range(frame_count):
        success, img = cap.read()
        if not success:
            break

        # Apply mask
        if mask is not None:
            imgRegion = cv2.bitwise_and(img, mask)
        else:
            imgRegion = img

        # Process YOLO results
        results = model(imgRegion, stream=True)

        detections = np.empty((0, 5))
        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                w, h = x2 - x1, y2 - y1
                conf = math.ceil((box.conf[0] * 100)) / 100
                cls = int(box.cls[0])

                if cls < len(classNames) and classNames[cls] == "car" and conf > 0.3:
                    detections = np.vstack((detections, [x1, y1, x2, y2, conf]))

        # Update tracker
        resultsTracker = tracker.update(detections)

        # Draw results and count vehicles
        for result in resultsTracker:
            x1, y1, x2, y2, id = map(int, result)
            cx, cy = x1 + (x2 - x1) // 2, y1 + (y2 - y1) // 2
            if limits[0] < cx < limits[2] and abs(cy - limits[1]) < 5:
                if id not in totalcount:
                    totalcount.append(id)

        # Update progress bar and frame
        progress.progress((frame_id + 1) / frame_count)
        count_placeholder.write(f"Current Count: {len(totalcount)}")

    # Display final count
    st.success(f"Total Vehicle Count: {len(totalcount)}")

    # Release resources
    cap.release()
"""

# Save the code to a Python file
with open("app.py", "w") as f:
    f.write(streamlit_code)

# Run Streamlit as a subprocess
process = subprocess.Popen(["streamlit", "run", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Print logs in the notebook (optional)
for line in iter(process.stdout.readline, b""):
    print(line.decode().strip())



You can now view your Streamlit app in your browser.

Local URL: http://localhost:8501
Network URL: http://10.37.6.40:8501



In [173]:
!pip install notebook ipywidgets


In [1]:
!pip install --upgrade ipywidgets

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------------------- -------- 1.8/2.3 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 8.3 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.6
    Uninstalling widgetsnbextension-3.6.6:
      Successfully uninstalled widgetsnbextension-3.6.6
  Attempting uninstall: jupyterlab-widgets
    Found existing installation: jupyterlab-widgets 1.0.0
    Uninstalling jupyterlab-widgets-1.0.0:
      Successfully uninstalled jupyterlab-widgets-1.0.0
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.8.1
    Uninstalling ipywidgets-7.8.1:
      Successfully uninstalled ipywidgets-7.8.1


In [3]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)